### Imports

In [1]:
from pathlib import Path
import pandas as pd

### Daten einlesen

In [2]:
df_categories = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/categories.csv")
df_customers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/customers.csv", dtype={"customer_id": "string"})
df_geolocation = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/geolocation.csv")
df_items = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/items.csv")
df_orders = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/orders.csv", dtype={"order_id": "string", "customer_id": "string"})
df_payments = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/payments.csv")
df_pilot_support = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/pilot_support.csv")
df_products = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/products.csv")
df_reviews = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/reviews.csv")
df_sellers = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/sellers.csv")
df_service_experiment = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_experiment.csv")
df_service_observations = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/service_observation.csv")

## 2.1 Daten vorbereiten

### a) Daten Qualität prüfen

1. Was erwarte ich, bzw. Was sagt eine Zeile aus?
2. Woran erkenne ich eine Zeile eindeutig?
3. Wo weichen meine Daten ab?
4. Was kann ich daruas begründet entscheiden?

In [16]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
# 1. Eine Zeile soll eine Bestellung darstellen.
print("Bestellungen insgesamt:", len(df_orders))

# 2. Ist die Bestellnummer vollständig und eindeutig?
ids = df_orders["order_id"]
print("Fehlende Bestellnummern:", ids.isna().sum())
print("Mehrfach vorkommende Bestellnummern:", ids[ids.duplicated()].nunique())

# 3. Passen Status und tatsächliches Lieferdatum zusammen?
geliefert = df_orders["order_status"].eq("delivered")
storniert = df_orders["order_status"].eq("canceled")
ist_vorhanden = df_orders["order_delivered_customer_date"].notna()
soll_vorhanden = df_orders["order_estimated_delivery_date"].notna()

print("Geliefert, aber ohne Lieferdatum:", (geliefert & ~ist_vorhanden).sum())
print("Storniert, aber mit Lieferdatum:", (storniert & ist_vorhanden).sum())

# 4. Welche Bestellungen eignen sich für den Vergleich von Ist- und Solltermin?
fuer_liefervergleich = geliefert & ist_vorhanden & soll_vorhanden   # orders nicht "bereinigt", nur Zeilen mit vollständigen und konsistenten Daten zwischengespeichert
print("Geliefert und beide Termine vorhanden:", fuer_liefervergleich.sum())

Bestellungen insgesamt: 99441
Fehlende Bestellnummern: 0
Mehrfach vorkommende Bestellnummern: 0
Geliefert, aber ohne Lieferdatum: 8
Storniert, aber mit Lieferdatum: 6
Geliefert und beide Termine vorhanden: 96470


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine Bestellung. Bei einer zugestellten Bestellung erwarte ich ein tatsächliches Lieferdatum.
2. **Was macht die Zeile eindeutig?** Die `order_id` ist bei allen 99.441 Bestellungen vorhanden und eindeutig.
3. **Wo weichen die Daten ab?** Bei 8 als zugestellt markierten Bestellungen fehlt das Lieferdatum. Bei 6 stornierten Bestellungen ist ein Lieferdatum eingetragen.
4. **Was entscheide ich?** Für den Vergleich von zugesagtem und tatsächlichem Liefertermin verwende ich die 96.470 zugestellten Bestellungen, bei denen beide Termine vorliegen. Für eine Korrektur der auffälligen Einträge fehlt ein Beleg.

In [18]:
df_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ce02b2c6e70d5b2fe24d103952d389bf,1,70d17ed46123e5b68dff23e113415d5a,3d871de0142ce09b7081e2b9d1733cb1,2017-08-03 13:45:11,54.90,12.72
1,4886c605fd1f7edd568a936814a015b7,1,bb09b0b5f3668effd704b1b2a83cae9b,6560211a19b47992c3666cc44a7e94c0,2017-12-22 02:31:10,189.00,8.58
2,2abac84637c5e0ddd2923f22ba61a96a,1,4ae634441e444ca4bc85903cafe98d73,92eb0f42c21942b6552362b9b114707d,2017-04-05 01:22:35,25.00,16.05
3,906dfb858a6afcaea1fb5cc0ed730147,1,33e77eecc23c25a65d10f627e5053617,7c67e1448b00f6e969d365cea6b010ab,2017-11-01 02:49:41,114.94,28.16
4,dcc495566353215aa81cab568f8fb2bf,1,6fd08d44046ab994b96ff38ad6fcfba1,d9a84e1403de8da0c3aa531d6d108ba6,2018-03-30 16:08:15,39.90,18.23


In [19]:
schluessel = ["order_id", "order_item_id"]

print("Positionszeilen insgesamt:", len(df_items))
print("Zeilen mit unvollständigem Schlüssel:", df_items[schluessel].isna().any(axis=1).sum())
print("Vollständig identische Wiederholungen:", df_items.duplicated().sum())

# Nur vollständig identische Zeilen entfernen
df_items_clean = df_items.drop_duplicates().copy()

print("Positionen nach Bereinigung:", len(df_items_clean))
print("Verbleibende Schlüsselwiederholungen:", df_items_clean.duplicated(schluessel).sum())

# Auswirkung auf den gesamten Warenwert zeigen
print("Warenwert vorher (BRL):", round(df_items["price"].sum(), 2))
print("Warenwert nachher (BRL):", round(df_items_clean["price"].sum(), 2))

Positionszeilen insgesamt: 112875
Zeilen mit unvollständigem Schlüssel: 0
Vollständig identische Wiederholungen: 225
Positionen nach Bereinigung: 112650
Verbleibende Schlüsselwiederholungen: 0
Warenwert vorher (BRL): 13619897.04
Warenwert nachher (BRL): 13591643.7


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine Position innerhalb einer Bestellung mit Produkt, Händler, Preis und Frachtkosten.
2. **Was macht die Zeile eindeutig?** Die Kombination aus Bestellnummer (`order_id`) und Positionsnummer (`order_item_id`).
3. **Wo weichen die Daten ab?** Unter den 112.875 Positionszeilen gibt es 225 vollständig identische Wiederholungen. Beim Summieren würden ihre Preise und Frachtkosten doppelt zählen.
4. **Was entscheide ich?** Ich entferne die identischen Wiederholungen. Es bleiben 112.650 Positionen mit eindeutigem Schlüssel. Für die weitere Analyse verwende ich `df_items_clean`.

In [20]:
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [21]:
ids = df_customers["customer_id"]

print("Kundeneinträge:", len(df_customers))
print("Fehlende customer_id:", ids.isna().sum())
print("Wiederholte customer_id:", ids.dropna().duplicated().sum())
print("Fehlender Bundesstaat:", df_customers["customer_state"].isna().sum())
print("Bestellungen ohne passenden Kundeneintrag:",
      (~df_orders["customer_id"].isin(ids.dropna())).sum())

Kundeneinträge: 99441
Fehlende customer_id: 0
Wiederholte customer_id: 0
Fehlender Bundesstaat: 0
Bestellungen ohne passenden Kundeneintrag: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt den Kundeneintrag zu einer Bestellung. Dieselbe Person kann bei mehreren Bestellungen vorkommen.
2. **Was macht die Zeile eindeutig?** Die `customer_id`. Die `customer_unique_id` erkennt dieselbe Person über mehrere Bestellungen hinweg und darf sich wiederholen.
3. **Wo weichen die Daten ab?** Bei den geprüften Feldern gibt es keine Auffälligkeiten. Jede `customer_id` ist vorhanden und eindeutig, jeder Eintrag enthält einen Bundesstaat und jede Bestellung hat einen passenden Kundeneintrag.
4. **Was entscheide ich?** Für diese Felder ist keine Bereinigung nötig. Ich verwende die `customer_id`, um jeder Bestellung den Bundesstaat des Kunden zuzuordnen.

In [22]:
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [23]:
ids = df_sellers["seller_id"]

print("Händlereinträge:", len(df_sellers))
print("Fehlende seller_id:", ids.isna().sum())
print("Wiederholte seller_id:", ids.dropna().duplicated().sum())
print("Fehlender Bundesstaat:", df_sellers["seller_state"].isna().sum())
print("Positionen ohne passenden Händler:",
      (~df_items_clean["seller_id"].isin(ids.dropna())).sum())

Händlereinträge: 3095
Fehlende seller_id: 0
Wiederholte seller_id: 0
Fehlender Bundesstaat: 0
Positionen ohne passenden Händler: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt einen Händler und seinen Standort.
2. **Was macht die Zeile eindeutig?** Die `seller_id` identifiziert jeden Händler eindeutig.
3. **Wo weichen die Daten ab?** Alle Händler-IDs sind vorhanden und eindeutig. Jeder Händler hat einen eingetragenen Bundesstaat, und jede Bestellposition lässt sich einem Händler zuordnen.
4. **Was entscheide ich?** Für diese Felder ist keine Bereinigung nötig. Ich nutze `seller_id`, um den Händlerstandort jeder Position mit dem Bundesstaat des Kunden zu vergleichen.

In [24]:
df_reviews.head()

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53


In [25]:
anzahl = df_reviews.groupby("order_id").size()

print("Bewertungszeilen:", len(df_reviews))
print("Zeilen ohne Bestellnummer:", df_reviews["order_id"].isna().sum())
print("Bestellungen mit mehreren Bewertungen:", (anzahl > 1).sum())
print("Fehlende Abgabezeitpunkte:", df_reviews["review_answer_timestamp"].isna().sum())
print("Fehlende oder ungültige Sternezahl:",
      (~df_reviews["review_score"].isin([1, 2, 3, 4, 5])).sum())

Bewertungszeilen: 99224
Zeilen ohne Bestellnummer: 0
Bestellungen mit mehreren Bewertungen: 547
Fehlende Abgabezeitpunkte: 0
Fehlende oder ungültige Sternezahl: 0


1. **Was sagt eine Zeile aus?** Eine Zeile beschreibt eine abgegebene Bewertung zu einer Bestellung.
2. **Was macht die Zeile eindeutig?** Einen eindeutigen Einzelschlüssel gibt die Aufgabenstellung für diese Tabelle nicht vor. Die `order_id` darf mehrfach vorkommen, und die `review_id` ist nicht global eindeutig.
3. **Wo weichen die Daten ab?** Bei 547 Bestellungen liegen mehrere Bewertungen vor. Das ist zulässig. Bei allen 99.224 Bewertungszeilen sind Bestellnummer und Abgabezeitpunkt vorhanden.
4. **Was entscheide ich?** In 2.1b wähle ich je Bestellung die Bewertung mit dem spätesten Abgabezeitpunkt. Bei Gleichstand nehme ich die größere `review_id`. Anschließend prüfe ich, ob diese Bewertung 1 bis 5 Sterne hat.